In [ ]:
from langchain.agents import craeate_agent
from lanchain.tools import tool, ToolRuntime
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import Command
from pydantic import BaseModel
from dataclasses import dataclass
from langchain.agents.middleware import SummarizationMiddleware

system_prompt=""


@dataclass
class fav_col:
    fav_col: str = 'nothing'

@tools
def get_fav_col(runtime: ToolRuntime) -> str:
    """Get the favourite colour of the useer"""
    return runtime.context.favourite_colour

@tools 
def update_fav_col(favourite_colour:str, runtime:ToolRuntime) -> Command:
    """Update the favourite colour of the user in the state they have revealed it"""
    return Command(update={
        "favourite_colour": favourite_colour,
        "messages": [ToolMessage("Succesfully updated favourite colour", tool_call_id=runtime.tool_call_id)]})

@before_agent
def trim_message(state:AgentState, runtime:Runtime) -> dict[str, Any] | None:
    """Remove all the tool messages from the state"""
    messages = state["messages"]
    tool_messages = [m for m in messages if isinstance(m, HumanMessage)]
    return {"messages": [RemoveMessage(id=m.id) for m in tool_messages]}

class Ticket(BaseModel):

agent = craeate_agent(model='claude-sonnet-4-6',
                      system_prompt=system_prompt,
                      tools = ,
                      checkpointe=InMemorySaver(),
                      middleware=[
                          trim_messages
                      ]
                      context_schema=ColourContext
                      response_format=Ticket)

In [ ]:
from langchain_mcp_adapters.client import MultiServerMCPClient

client = MultiServerMCPClient(
    {
        "local_server": {
            "transport": "streamable_http",
            "command": "python",
        }
    }
)

In [ ]:
tools = await client.get_tools()
resources = await client.get_resources("local_server")
prompt = await client.get_prompt("local_server", "prompt")
prompt = prompt[0].content